In [13]:
import cobra 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns   
from cobra.flux_analysis import loopless_solution


# metabolic modeling of cyanobacteria - FBA
## Goal : compare metabolism of (i) WT cyanobacteria, (ii) a mutant with native phosphorespiration pathway deleted (dCCM), and (iii) the mutant dCCM with BAHC reactions added (pathway to recycle C2 compound produced by rubisco)
### We will use the previously published metabolic network iMS837 to simulate fluxes in the WT, and modify the network to model the (ii) and (iii) cases


In this notebook we predict the fluxes for the WT

In [14]:
BASE_FOLDER='/home/nathan/PhD/04.Computational/202609_BHAC_cyano/'
INPUT_FOLDER=f'{BASE_FOLDER}1_data/'

Load cobra model 

In [15]:
model = cobra.io.load_json_model(f'{INPUT_FOLDER}iMS837.json')

Set bounds on exchange reactions to simulate the proper experimental conditions

In [16]:
### nitrate is the only N-source -> no ammonium/ammonia
model.reactions.EX_nh4_e.lower_bound = 0
## we will constrain atmospheric CO2 in %, so we block hco3 uptake 
model.reactions.EX_hco3_e.lower_bound = 0

In [20]:
model.reactions.EX_co2_e

Reaction identifier,EX_co2_e
Name,Modeling: exchange reaction (co2_e)
Memory address,0x79256b4356f0
Stoichiometry,co2_e <=> CO2 <=>
GPR,
Lower bound,-1.99
Upper bound,1000.0


In [17]:
###biomass is model.reactions.BOF
model.objective='BOF'

In [18]:
###loopless FBA

solution = model.optimize()          # regular FBA first, to get the objective
sol = loopless_solution(model, fluxes=solution.fluxes)  # remove thermodynamically infeasible internal loops
dfsol=sol.fluxes.to_frame()

In [24]:
co2_levels = {'1pct': 1.0, '0.5pct': 0.5, '0.3pct': 0.3}
reference_pct = 0.04          # e.g. ambient air, your calibration point
reference_bound = 2.0         # mmol/gDW/h — the CO2 uptake bound you've validated growth against at reference_pct

for label, pct in co2_levels.items():
    scaled_bound = reference_bound * (pct / reference_pct)
    with model:  # context manager: change reverts after the block
        model.exchanges.get_by_id('EX_co2_e').lower_bound = -scaled_bound
        sol = model.optimize()
        print(label, '-> CO2 bound:', -scaled_bound, '| growth:', sol.objective_value)

1pct -> CO2 bound: -50.0 | growth: 0.21873550959094956
0.5pct -> CO2 bound: -25.0 | growth: 0.21873550959094956
0.3pct -> CO2 bound: -15.0 | growth: 0.21873550959094956


analyze flux of reactions that have at least one metabolite in the carboxysome 

In [8]:
rxn_cx=list(set([r for r in model.reactions for m in r.metabolites if 'carboxyzome' in m.compartment ]))
dfsol_cx=dfsol.loc[[r.id for r in rxn_cx]]

In [9]:
### glyclt_c is glycolate , ### RBPCcx is rubisco reaction
## we are interested in the fluxes that consume produces the following metabolites 
## glycine gly_c, serine ser__L_c, aspartate asp__L_c
rxn_13cmets=list(set([r for r in model.reactions for m in r.metabolites if m.id in ['gly_c','ser__L_c','asp__L_c']]))

In [10]:
dfsol_13cmets=dfsol.loc[[r.id for r in rxn_13cmets]]
dfsol_13cmets.drop

<bound method DataFrame.drop of                  fluxes
PSP_L      2.294306e-01
AMPTASECG  0.000000e+00
ASPTA     -9.087121e-02
GLYDHDA    0.000000e+00
TRPS2      3.641648e-03
ADSS       8.682725e-03
ASNS1      9.528719e-03
TRPS1      0.000000e+00
PRASCSi    9.649613e-03
PRAGSr     9.650908e-03
ARGSS      1.184708e-02
ASPO5      0.000000e+00
SERAT      2.348084e-03
GTHS       0.000000e+00
ASPO6      1.324069e-05
ASNN       0.000000e+00
GLYCOX1    1.294889e-06
GLYTRS     1.735644e-02
AGTi       2.079010e-02
SPTr      -1.924085e-01
ASPCT      9.625703e-03
GHMT2r     1.738790e-02
ASPK       2.969952e-02
ASP1DC     9.726862e-07
GLYCL      1.116936e-02
ASPTRS     1.182364e-02
SERTRS     1.364446e-02>

In [11]:
for r in rxn_cx:
    print(r.id,r.reaction)

RBPCcx 0.99 co2_cx + 0.99 h2o_cx + 0.009999999999999998 o2_cx + rb15bp_cx --> 0.009999999999999998 2pglyc_cx + 1.99 3pg_cx + 2.0 h_cx
2PGLYCtcx 2pglyc_cx --> 2pglyc_c
3PGtcx 3pg_cx --> 3pg_c
H2Otcx h2o_c <=> h2o_cx
Htcx h_c <=> h_cx
RB15BPtcx rb15bp_c --> rb15bp_cx
HCO3tcx hco3_c --> hco3_cx
HCO3E_1_cx h_cx + hco3_cx <=> co2_cx + h2o_cx
O2tcx o2_c --> o2_cx


In [12]:
model.metabolites.get_by_id('rb15bp_c')
# model.reactions.get_by_id('HCO3E_c')

Metabolite identifier,rb15bp_c
Name,"D-Ribulose 1,5-bisphosphate"
Memory address,0x79256ca1d870
Formula,C5H8O11P2
Compartment,cytosol
In 2 reaction(s),"PRUK, RB15BPtcx"
